### Setup

In [5]:
import pandas as pd
import os

from neo4j import GraphDatabase
from decimal import Decimal
from dotenv import load_dotenv

load_dotenv("../.env")

EDGES_DIR = "../data/edges/"

class Config:
    def __init__(self, mode="LOCAL"):
        mode = mode.upper()

        if mode == "LOCAL":
            self.URI = os.getenv("NEO4J_URI_LOCAL")
            self.USER = os.getenv("NEO4J_USER_LOCAL")
            self.PASSWORD = os.getenv("NEO4J_PASSWORD_LOCAL")
        elif mode == "GROUP":
            self.URI = os.getenv("NEO4J_GROUP_URI")
            self.USER = os.getenv("NEO4J_GROUP_USER")
            self.PASSWORD = os.getenv("NEO4J_GROUP_PASSWORD")
        else:
            raise ValueError("Mode must be 'LOCAL' or 'GROUP'.")

        self.DATABASE = "neo4j"

config = Config(mode="LOCAL")
driver = GraphDatabase.driver(config.URI, auth=(config.USER, config.PASSWORD))

with driver.session(database=config.DATABASE) as session:
    session.run("MATCH (n) RETURN n LIMIT 1")
    print(f"Connection successful")

def query_neo4j(cypher_query: str, parameters: dict = None):
    with driver.session(database=config.DATABASE) as session:
        result = session.run(cypher_query, parameters)
        return result.data()

Connection successful


In [6]:
def get_relationship_count(label1, label2, relationship):
    result = query_neo4j(f"MATCH (n1:`{label1}`)-[r]-(n2:`{label2}`) RETURN DISTINCT TYPE(r) AS relationship, COUNT(*) as count")
    relationship_counts = {(i["relationship"]): i["count"] for i in result}
    return relationship_counts.get(relationship, 0)

def delete_relationships(label1, label2, relationship):
    existing_count = get_relationship_count(label1, label2, relationship)
    if existing_count > 0:
        query_neo4j(f"MATCH (a:`{label1}`)-[r:{relationship}]->(b:`{label2}`) DELETE r")
        new_count = get_relationship_count(label1, label2, relationship)
        assert new_count == 0, f"All {relationship} relationships between {label1} and {label2} were not deleted"
    print(f"Deleted {existing_count} `{relationship}` relationships between {label1} and {label2}.")

def create_relationships(label1, label2, relationship, data):
    query = f"""
    UNWIND $data AS row
    MATCH (a:`{label1}` {{id: row.id1}})
    MATCH (b:`{label2}` {{id: row.id2}})
    MERGE (a)-[r:{relationship}]->(b)
    """
    query_neo4j(query, parameters={"data": data})
    new_count = get_relationship_count(label1, label2, relationship)
    print(f"Created {new_count} `{relationship}` relationships between {label1} and {label2}.")

def load_relationships(json_file):
    df = pd.read_json(EDGES_DIR + json_file)
    label1 = df.iloc[0]["entitytype1"]
    label2 = df.iloc[0]["entitytype2"]
    relationship = df.iloc[0]["predicate"]
    data = [{"id1": int(row["entity1"]), "id2": int(row["entity2"])} for _, row in df.iterrows()]
    delete_relationships(label1, label2, relationship)
    create_relationships(label1, label2, relationship, data)

def view_relationships():
    result = query_neo4j("""
        MATCH (n1)-[r]-(n2)
        RETURN DISTINCT
            labels(n1) AS label1,
            type(r) AS relationship,
            labels(n2) AS label2,
            count(*) AS count
    """)
    
    data = [
        {
            "label1": record["label1"],
            "relationship": record["relationship"],
            "label2": record["label2"],
            "count": record["count"]
        }
        for record in result
    ]

    df = pd.DataFrame(data)
    return df

### Load Nodes From Json

In [7]:
relationship_files = os.listdir(EDGES_DIR)
relationship_files

['businesslocation_belongs_to_business.json',
 'county_adjacent_to_county.json',
 'city_adjacent_to_city.json',
 'city_nearby_city.json',
 'businesslocation_contained_in_city.json',
 'county_contained_in_state.json',
 'community_contained_in_city.json',
 'city_contained_in_county.json',
 'community_adjacent_to_community.json',
 'community_nearby_community.json',
 'community_overlaps_blockgroup.json',
 'community_overlaps_zipcode.json',
 'businesslocation_contained_in_zipcode.json',
 'businesslocation_contained_in_blockgroup.json',
 'zonelocation_belongs_to_zonetype.json']

In [8]:
for file in relationship_files:
    print(f"\nLoading relationships from file: {file}")
    load_relationships(file)


Loading relationships from file: businesslocation_belongs_to_business.json
Deleted 39593 `BELONGS_TO` relationships between BusinessLocation and Business.
Created 39593 `BELONGS_TO` relationships between BusinessLocation and Business.

Loading relationships from file: county_adjacent_to_county.json
Deleted 10 `ADJACENT_TO` relationships between County and County.
Created 10 `ADJACENT_TO` relationships between County and County.

Loading relationships from file: city_adjacent_to_city.json
Deleted 146 `ADJACENT_TO` relationships between City and City.
Created 146 `ADJACENT_TO` relationships between City and City.

Loading relationships from file: city_nearby_city.json
Deleted 286 `NEARBY` relationships between City and City.
Created 286 `NEARBY` relationships between City and City.

Loading relationships from file: businesslocation_contained_in_city.json
Deleted 38293 `CONTAINED_IN` relationships between BusinessLocation and City.
Created 38293 `CONTAINED_IN` relationships between Busin

### Add Nodes Using Cypher ( DO NOT USE THIS YET. IT NEEDS TO BE OPTIMIZED)

In [ ]:
#Business Locations - shared Block Group - Business Location
  

apoc_query = """
CALL apoc.periodic.iterate(
    "MATCH (bl1:BusinessLocation)-[:CONTAINED_IN]->(blockgroup:BlockGroup)<-[:CONTAINED_IN]-(bl2:BusinessLocation)
    WHERE elementId(bl1) < elementId(bl2)
    AND bl1.location IS NOT NULL AND bl2.location IS NOT NULL
    RETURN bl1, bl2, blockgroup",

    

"WITH bl1, bl2, blockgroup
    MERGE (bl1)-[s:SHARED_REGION]->(bl2)
    ON CREATE SET 
        s.shared_blockgroup_id = blockgroup.ctblockgroup,
        s.shares_blockgroup = true,
        s.distance_meters = point.distance(bl1.location, bl2.location)
    ON MATCH SET 
        s.shared_blockgroup_id = blockgroup.ctblockgroup,
        s.shares_blockgroup = true",

{batchSize: 10000, parallel: true}
)
YIELD batches, total, timeTaken, committedOperations
RETURN batches, total, timeTaken, committedOperations

"""
with driver.session(database=config.DATABASE) as session:
    # Run the APOC query
    apoc_result = session.run(apoc_query)
    
    print("APOC Query executed successfully and work was batched.")
    
    # Print the batching results for confirmation
    for record in apoc_result:
        print(f"* Total operations committed: {record['committedOperations']}")
        print(f"* Total time taken: {record['timeTaken']} ms")
        print(f"* Batches completed: {record['batches']}")


# with driver.session(database=config.DATABASE) as session:

#     shared_blockgroup= session.run( query)
#     summary = shared_blockgroup.consume()
#     print(f" Query executed successfully. Statement type: {summary.statement_type}")


✅ APOC Query executed successfully and work was batched.


ResultConsumedError: The result has been consumed. Fetch all needed records before calling Result.consume().

In [10]:
# Business Location - shared City - Business Location 

apoc_city_query = """
CALL apoc.periodic.iterate(
    "MATCH (bl1:BusinessLocation)-[:CONTAINED_IN]->(city:City)<-[:CONTAINED_IN]-(bl2:BusinessLocation)
    WHERE elementId(bl1) < elementId(bl2)
    AND bl1.location IS NOT NULL AND bl2.location IS NOT NULL
    RETURN bl1, bl2, city",

    "WITH bl1, bl2, city
    MERGE (bl1)-[s:SHARED_REGION]->(bl2)
    ON CREATE SET 
        s.shared_city_name = city.name,
        s.shares_city = true,
        s.distance_meters = point.distance(bl1.location, bl2.location)
    ON MATCH SET 
        s.shared_city_name = city.name,
        s.shares_city = true",
    
    {batchSize: 10000, parallel: true}
)
YIELD batches, total, timeTaken, committedOperations
RETURN batches, total, timeTaken, committedOperations
"""


with driver.session(database=config.DATABASE) as session:
    # 1. Run the APOC query
    apoc_result = session.run(apoc_city_query)
    
    print("APOC Query starting, operations will be batched.")
    
    # 2. Iterate to get the single APOC summary record
    for record in apoc_result:
        print(f"**Query Execution Summary:**")
        print(f"* Total MERGE operations committed: {record['committedOperations']}")
        print(f"* Total time taken: {record['timeTaken']} ms")
        print(f"* Batches completed: {record['batches']}")
        
    

APOC Query starting, operations will be batched.
**Query Execution Summary:**
* Total MERGE operations committed: 26550000
* Total time taken: 2523 ms
* Batches completed: 22292


In [11]:
query = """
MATCH (bl1:BusinessLocation)-[s:SHARED_REGION]-(bl2:BusinessLocation)
WHERE elementId(bl1) < elementId(bl2)

WITH s,
    CASE WHEN s.shared_city_name IS NOT NULL THEN 1 ELSE 0 END AS city_count,
    CASE WHEN s.shared_blockgroup_id IS NOT NULL THEN 1 ELSE 0 END AS blockgroup_count,
    CASE WHEN s.shared_zipcode_id IS NOT NULL THEN 1 ELSE 0 END AS zip_count
WITH s, (city_count + blockgroup_count + zip_count) AS calculated_weight
SET s.weight = calculated_weight
RETURN count(s) AS relationships_updated"""


with driver.session(database=config.DATABASE) as session:

# with group_driver.session() as session:
    result = session.run( query)